# NNUE

## CONFIGURATION

### Imports

In [47]:
import numpy as np
import pandas as pd
import chess
from scipy import sparse

### Constants

In [48]:
INPUT_COUNT = 768
LAYER_1_SIZE = 512
LAYER_2_SIZE = 256
LAYER_3_SIZE = 128
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 0.001  # Reduced from 0.5
BETA1 = 0.9
BETA2 = 0.999
EPSILON = 1e-8

### Key Functions
(Note: Regenerate dataset if labels contain '#' mate scores, as previous logic treated mate as 0.0 eval)

In [49]:
def ReLU(x):
    return np.maximum(0, x)

def Sigmoid(x):
    return 1 / (1 + np.exp(-x))

def ReLU_derivative(x):
    return np.where(x > 0, 1, 0)

def Sigmoid_derivative(x, value = None):
    if (value is not None):
        return value * (1 - value)

    s = Sigmoid(x)
    return s * (1 - s)

# def centipawns_to_win_probability(centipawns : str) -> float:
#     return 1 / (1 + np.exp(-int(centipawns.replace('#', '')) / 400))

def centipawns_to_win_probability(centipawns : str) -> float:
    if '#' in centipawns:
        return 1.0 if '+' in centipawns else 0.0
    return 1 / (1 + np.exp(-int(centipawns) / 150))

def FEN_to_input_vector(fen : str) -> np.ndarray:
    board = chess.Board(fen)
    turn = board.turn 
    
    input_vector = np.zeros(768, dtype=np.float32) # Moins de RAM !
        
    # Optimisation : piece_map() ne boucle que sur les cases occupées (max 32) au lieu de 64
    for square, piece in board.piece_map().items():
        display_square = square if turn == chess.WHITE else chess.square_mirror(square)
        
        piece_type = piece.piece_type - 1
        # Si c'est la pièce du joueur dont c'est le tour, offset = 0, sinon 6
        color_offset = 0 if piece.color == turn else 6               
                    
        input_vector[display_square * 12 + color_offset + piece_type] = 1   
            
    return input_vector

## DATASET 

### Basic Import

In [50]:
labels = np.load('labels.npz')["Y"]
moves = sparse.load_npz('moves.npz')

## Model

### Weights

In [51]:
W1 = np.random.randn(INPUT_COUNT, LAYER_1_SIZE) * np.sqrt(2.0 / INPUT_COUNT)
b1 = np.zeros((1, LAYER_1_SIZE))
W2 = np.random.randn(LAYER_1_SIZE, LAYER_2_SIZE) * np.sqrt(2.0 / LAYER_1_SIZE)
b2 = np.zeros((1, LAYER_2_SIZE))
W3 = np.random.randn(LAYER_2_SIZE, LAYER_3_SIZE) * np.sqrt(2.0 / LAYER_2_SIZE)
b3 = np.zeros((1, LAYER_3_SIZE))
W4 = np.random.randn(LAYER_3_SIZE, 1) * np.sqrt(1.0 / LAYER_3_SIZE)
b4 = np.zeros((1, 1))

vW1, vb1 = np.zeros_like(W1), np.zeros_like(b1)
vW2, vb2 = np.zeros_like(W2), np.zeros_like(b2)
vW3, vb3 = np.zeros_like(W3), np.zeros_like(b3)
vW4, vb4 = np.zeros_like(W4), np.zeros_like(b4)

mW1, mb1 = np.zeros_like(W1), np.zeros_like(b1)
mW2, mb2 = np.zeros_like(W2), np.zeros_like(b2)
mW3, mb3 = np.zeros_like(W3), np.zeros_like(b3)
mW4, mb4 = np.zeros_like(W4), np.zeros_like(b4)

### Forward Pass

In [52]:
def forward_pass(X, W1, b1, W2, b2, W3, b3, W4, b4):
    
    Z1 = np.dot(X, W1) + b1
    A1 = ReLU(Z1)
    
    Z2 = np.dot(A1, W2) + b2
    A2 = ReLU(Z2)
    
    Z3 = np.dot(A2, W3) + b3
    A3 = ReLU(Z3)
    
    Z4 = np.dot(A3, W4) + b4
    A4 = Sigmoid(Z4)
    
    return A1, Z1, A2, Z2, A3, Z3, A4, Z4

### Backward Pass

In [53]:
def backward_pass(input, W1, b1, A1, Z1, W2, b2, A2, Z2, W3, b3, A3, Z3, W4, b4, A4, Z4, labels):
    m = input.shape[0]

    # Couche 4
    dZ4 = A4 - labels
    dW4 = np.dot(A3.T, dZ4) / m
    db4 = np.sum(dZ4, axis=0, keepdims=True) / m

    # Couche 3
    dA3 = np.dot(dZ4, W4.T)
    dZ3 = dA3 * ReLU_derivative(Z3)
    dW3 = np.dot(A2.T, dZ3) / m  
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m
    
    # Couche 2
    dA2 = np.dot(dZ3, W3.T)
    dZ2 = dA2 * ReLU_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / m  
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    
    # Couche 1
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * ReLU_derivative(Z1)
    dW1 = np.dot(input.T, dZ1) / m  
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return dW1, db1, dW2, db2, dW3, db3, dW4, db4

### Training

In [54]:
t = 0

for epoch in range(EPOCHS):
    indices = np.arange(moves.shape[0])
    np.random.shuffle(indices)
    total_loss = 0
    
    # Using batches
    for i in range(0, len(indices), BATCH_SIZE):
        t+= 1
        batch_indices = indices[i : i + BATCH_SIZE]
        
        # Sparse to dense just for batch
        X_batch = moves[batch_indices].toarray() 
        y_batch = labels[batch_indices].reshape(-1, 1)
        
        # Forward pass (Explicitly passing weights)
        A1, Z1, A2, Z2, A3, Z3, A4, Z4 = forward_pass(X_batch, W1, b1, W2, b2, W3, b3, W4, b4)
        # Loss calculation
        epsilon = 1e-15
        loss = -np.mean(y_batch * np.log(A4 + epsilon) + (1 - y_batch) * np.log(1 - A4 + epsilon))
        total_loss += loss
        
        if (i // BATCH_SIZE) % 100 == 0:
            print(f"Epoch {epoch}, Batch {i//BATCH_SIZE}: Loss: {loss:.4f}")
        
        # Backward pass
        dW1, db1, dW2, db2, dW3, db3, dW4, db4 = backward_pass(
            X_batch, W1, b1, A1, Z1, W2, b2, A2, Z2, W3, b3, A3, Z3, W4, b4, A4, Z4, y_batch
        )
        # Adam update
        mW1 = BETA1 * mW1 + (1 - BETA1) * dW1
        mb1 = BETA1 * mb1 + (1 - BETA1) * db1
        mW2 = BETA1 * mW2 + (1 - BETA1) * dW2
        mb2 = BETA1 * mb2 + (1 - BETA1) * db2
        mW3 = BETA1 * mW3 + (1 - BETA1) * dW3
        mb3 = BETA1 * mb3 + (1 - BETA1) * db3 
        mW4 = BETA1 * mW4 + (1 - BETA1) * dW4
        mb4 = BETA1 * mb4 + (1 - BETA1) * db4
        
        vW1 = BETA2 * vW1 + (1 - BETA2) * np.square(dW1)
        vb1 = BETA2 * vb1 + (1 - BETA2) * np.square(db1)
        vW2 = BETA2 * vW2 + (1 - BETA2) * np.square(dW2)
        vb2 = BETA2 * vb2 + (1 - BETA2) * np.square(db2)
        vW3 = BETA2 * vW3 + (1 - BETA2) * np.square(dW3)
        vb3 = BETA2 * vb3 + (1 - BETA2) * np.square(db3)
        vW4 = BETA2 * vW4 + (1 - BETA2) * np.square(dW4)
        vb4 = BETA2 * vb4 + (1 - BETA2) * np.square(db4)
        
        mW1_hat = mW1 / (1 - BETA1 ** t)
        mb1_hat = mb1 / (1 - BETA1 ** t)
        mW2_hat = mW2 / (1 - BETA1 ** t)
        mb2_hat = mb2 / (1 - BETA1 ** t)
        mW3_hat = mW3 / (1 - BETA1 ** t)
        mb3_hat = mb3 / (1 - BETA1 ** t)
        mW4_hat = mW4 / (1 - BETA1 ** t)
        mb4_hat = mb4 / (1 - BETA1 ** t)

        vW1_hat = vW1 / (1 - BETA2 ** t)
        vb1_hat = vb1 / (1 - BETA2 ** t)
        vW2_hat = vW2 / (1 - BETA2 ** t)
        vb2_hat = vb2 / (1 - BETA2 ** t)
        vW3_hat = vW3 / (1 - BETA2 ** t)
        vb3_hat = vb3 / (1 - BETA2 ** t)
        vW4_hat = vW4 / (1 - BETA2 ** t)
        vb4_hat = vb4 / (1 - BETA2 ** t)

        # Update weights
        W1 -= LEARNING_RATE / (np.sqrt(vW1_hat) + EPSILON) * mW1_hat
        b1 -= LEARNING_RATE / (np.sqrt(vb1_hat) + EPSILON) * mb1_hat
        W2 -= LEARNING_RATE / (np.sqrt(vW2_hat) + EPSILON) * mW2_hat
        b2 -= LEARNING_RATE / (np.sqrt(vb2_hat) + EPSILON) * mb2_hat
        W3 -= LEARNING_RATE / (np.sqrt(vW3_hat) + EPSILON) * mW3_hat
        b3 -= LEARNING_RATE / (np.sqrt(vb3_hat) + EPSILON) * mb3_hat
        W4 -= LEARNING_RATE / (np.sqrt(vW4_hat) + EPSILON) * mW4_hat
        b4 -= LEARNING_RATE / (np.sqrt(vb4_hat) + EPSILON) * mb4_hat
    
    avg_loss = total_loss / (len(indices) // BATCH_SIZE)
    print(f"End of Epoch {epoch}, Loss: {avg_loss:.4f}")

Epoch 0, Batch 0: Loss: 0.7001
Epoch 0, Batch 100: Loss: 0.6904
Epoch 0, Batch 200: Loss: 0.6839
Epoch 0, Batch 300: Loss: 0.6845
Epoch 0, Batch 400: Loss: 0.6705
Epoch 0, Batch 500: Loss: 0.6893
Epoch 0, Batch 600: Loss: 0.6816
Epoch 0, Batch 700: Loss: 0.6839
Epoch 0, Batch 800: Loss: 0.6797
Epoch 0, Batch 900: Loss: 0.6796
Epoch 0, Batch 1000: Loss: 0.6811
Epoch 0, Batch 1100: Loss: 0.6759
Epoch 0, Batch 1200: Loss: 0.6839
Epoch 0, Batch 1300: Loss: 0.6934
Epoch 0, Batch 1400: Loss: 0.6855
Epoch 0, Batch 1500: Loss: 0.6810
Epoch 0, Batch 1600: Loss: 0.6773
Epoch 0, Batch 1700: Loss: 0.6787
Epoch 0, Batch 1800: Loss: 0.6629
Epoch 0, Batch 1900: Loss: 0.6669
Epoch 0, Batch 2000: Loss: 0.6694
Epoch 0, Batch 2100: Loss: 0.6831
Epoch 0, Batch 2200: Loss: 0.6725
Epoch 0, Batch 2300: Loss: 0.6743
Epoch 0, Batch 2400: Loss: 0.6706
Epoch 0, Batch 2500: Loss: 0.6756
Epoch 0, Batch 2600: Loss: 0.6849
Epoch 0, Batch 2700: Loss: 0.6942
Epoch 0, Batch 2800: Loss: 0.6755
Epoch 0, Batch 2900: Loss:

KeyboardInterrupt: 

## Full Code

In [ ]:
import numpy as np
import pandas as pd
import chess
from scipy import sparse

INPUT_COUNT = 768
LAYER_1_SIZE = 512
LAYER_2_SIZE = 256
LAYER_3_SIZE = 128
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 0.001  # Reduced from 0.5
BETA1 = 0.9
BETA2 = 0.999
EPSILON = 1e-8

def ReLU(x):
    return np.maximum(0, x)

def Sigmoid(x):
    return 1 / (1 + np.exp(-x))

def ReLU_derivative(x):
    return np.where(x > 0, 1, 0)

def Sigmoid_derivative(x, value = None):
    if (value is not None):
        return value * (1 - value)

    s = Sigmoid(x)
    return s * (1 - s)

# def centipawns_to_win_probability(centipawns : str) -> float:
#     return 1 / (1 + np.exp(-int(centipawns.replace('#', '')) / 400))

def centipawns_to_win_probability(centipawns : str) -> float:
    if '#' in centipawns:
        return 1.0 if '+' in centipawns else 0.0
    return 1 / (1 + np.exp(-int(centipawns) / 400))

def FEN_to_input_vector(fen : str) -> np.ndarray:
    board = chess.Board(fen)
    turn = board.turn 
    
    input_vector = np.zeros(768, dtype=np.float32) # Moins de RAM !
        
    # Optimisation : piece_map() ne boucle que sur les cases occupées (max 32) au lieu de 64
    for square, piece in board.piece_map().items():
        display_square = square if turn == chess.WHITE else chess.square_mirror(square)
        
        piece_type = piece.piece_type - 1
        # Si c'est la pièce du joueur dont c'est le tour, offset = 0, sinon 6
        color_offset = 0 if piece.color == turn else 6               
                    
        input_vector[display_square * 12 + color_offset + piece_type] = 1   
            
    return input_vector

labels = np.load('labels.npz')["Y"]
moves = sparse.load_npz('moves.npz')

W1 = np.random.randn(INPUT_COUNT, LAYER_1_SIZE) * np.sqrt(2.0 / INPUT_COUNT)
b1 = np.zeros((1, LAYER_1_SIZE))
W2 = np.random.randn(LAYER_1_SIZE, LAYER_2_SIZE) * np.sqrt(2.0 / LAYER_1_SIZE)
b2 = np.zeros((1, LAYER_2_SIZE))
W3 = np.random.randn(LAYER_2_SIZE, LAYER_3_SIZE) * np.sqrt(2.0 / LAYER_2_SIZE)
b3 = np.zeros((1, LAYER_3_SIZE))
W4 = np.random.randn(LAYER_3_SIZE, 1) * np.sqrt(1.0 / LAYER_3_SIZE)
b4 = np.zeros((1, 1))

vW1, vb1 = np.zeros_like(W1), np.zeros_like(b1)
vW2, vb2 = np.zeros_like(W2), np.zeros_like(b2)
vW3, vb3 = np.zeros_like(W3), np.zeros_like(b3)
vW4, vb4 = np.zeros_like(W4), np.zeros_like(b4)

mW1, mb1 = np.zeros_like(W1), np.zeros_like(b1)
mW2, mb2 = np.zeros_like(W2), np.zeros_like(b2)
mW3, mb3 = np.zeros_like(W3), np.zeros_like(b3)
mW4, mb4 = np.zeros_like(W4), np.zeros_like(b4)

def forward_pass(X, W1, b1, W2, b2, W3, b3, W4, b4):
    
    Z1 = np.dot(X, W1) + b1
    A1 = ReLU(Z1)
    
    Z2 = np.dot(A1, W2) + b2
    A2 = ReLU(Z2)
    
    Z3 = np.dot(A2, W3) + b3
    A3 = ReLU(Z3)
    
    Z4 = np.dot(A3, W4) + b4
    A4 = Sigmoid(Z4)
    
    return A1, Z1, A2, Z2, A3, Z3, A4, Z4

def backward_pass(input, W1, b1, A1, Z1, W2, b2, A2, Z2, W3, b3, A3, Z3, W4, b4, A4, Z4, labels):
    m = input.shape[0]

    # Couche 4
    dZ4 = A4 - labels
    dW4 = np.dot(A3.T, dZ4) / m
    db4 = np.sum(dZ4, axis=0, keepdims=True) / m

    # Couche 3
    dA3 = np.dot(dZ4, W4.T)
    dZ3 = dA3 * ReLU_derivative(Z3)
    dW3 = np.dot(A2.T, dZ3) / m  
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m
    
    # Couche 2
    dA2 = np.dot(dZ3, W3.T)
    dZ2 = dA2 * ReLU_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2) / m  
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    
    # Couche 1
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * ReLU_derivative(Z1)
    dW1 = np.dot(input.T, dZ1) / m  
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return dW1, db1, dW2, db2, dW3, db3, dW4, db4

t = 0

for epoch in range(EPOCHS):
    indices = np.arange(moves.shape[0])
    np.random.shuffle(indices)
    total_loss = 0
    
    # Using batches
    for i in range(0, len(indices), BATCH_SIZE):
        t+= 1
        batch_indices = indices[i : i + BATCH_SIZE]
        
        # Sparse to dense just for batch
        X_batch = moves[batch_indices].toarray() 
        y_batch = labels[batch_indices].reshape(-1, 1)
        
        # Forward pass (Explicitly passing weights)
        A1, Z1, A2, Z2, A3, Z3, A4, Z4 = forward_pass(X_batch, W1, b1, W2, b2, W3, b3, W4, b4)
        # Loss calculation
        epsilon = 1e-15
        loss = -np.mean(y_batch * np.log(A4 + epsilon) + (1 - y_batch) * np.log(1 - A4 + epsilon))
        total_loss += loss
        
        if (i // BATCH_SIZE) % 100 == 0:
            print(f"Epoch {epoch}, Batch {i//BATCH_SIZE}: Loss: {loss:.4f}")
        
        # Backward pass
        dW1, db1, dW2, db2, dW3, db3, dW4, db4 = backward_pass(
            X_batch, W1, b1, A1, Z1, W2, b2, A2, Z2, W3, b3, A3, Z3, W4, b4, A4, Z4, y_batch
        )
        # Adam update
        mW1 = BETA1 * mW1 + (1 - BETA1) * dW1
        mb1 = BETA1 * mb1 + (1 - BETA1) * db1
        mW2 = BETA1 * mW2 + (1 - BETA1) * dW2
        mb2 = BETA1 * mb2 + (1 - BETA1) * db2
        mW3 = BETA1 * mW3 + (1 - BETA1) * dW3
        mb3 = BETA1 * mb3 + (1 - BETA1) * db3 
        mW4 = BETA1 * mW4 + (1 - BETA1) * dW4
        mb4 = BETA1 * mb4 + (1 - BETA1) * db4
        
        vW1 = BETA2 * vW1 + (1 - BETA2) * np.square(dW1)
        vb1 = BETA2 * vb1 + (1 - BETA2) * np.square(db1)
        vW2 = BETA2 * vW2 + (1 - BETA2) * np.square(dW2)
        vb2 = BETA2 * vb2 + (1 - BETA2) * np.square(db2)
        vW3 = BETA2 * vW3 + (1 - BETA2) * np.square(dW3)
        vb3 = BETA2 * vb3 + (1 - BETA2) * np.square(db3)
        vW4 = BETA2 * vW4 + (1 - BETA2) * np.square(dW4)
        vb4 = BETA2 * vb4 + (1 - BETA2) * np.square(db4)
        
        mW1_hat = mW1 / (1 - BETA1 ** t)
        mb1_hat = mb1 / (1 - BETA1 ** t)
        mW2_hat = mW2 / (1 - BETA1 ** t)
        mb2_hat = mb2 / (1 - BETA1 ** t)
        mW3_hat = mW3 / (1 - BETA1 ** t)
        mb3_hat = mb3 / (1 - BETA1 ** t)
        mW4_hat = mW4 / (1 - BETA1 ** t)
        mb4_hat = mb4 / (1 - BETA1 ** t)

        vW1_hat = vW1 / (1 - BETA2 ** t)
        vb1_hat = vb1 / (1 - BETA2 ** t)
        vW2_hat = vW2 / (1 - BETA2 ** t)
        vb2_hat = vb2 / (1 - BETA2 ** t)
        vW3_hat = vW3 / (1 - BETA2 ** t)
        vb3_hat = vb3 / (1 - BETA2 ** t)
        vW4_hat = vW4 / (1 - BETA2 ** t)
        vb4_hat = vb4 / (1 - BETA2 ** t)

        # Update weights
        W1 -= LEARNING_RATE / (np.sqrt(vW1_hat) + EPSILON) * mW1_hat
        b1 -= LEARNING_RATE / (np.sqrt(vb1_hat) + EPSILON) * mb1_hat
        W2 -= LEARNING_RATE / (np.sqrt(vW2_hat) + EPSILON) * mW2_hat
        b2 -= LEARNING_RATE / (np.sqrt(vb2_hat) + EPSILON) * mb2_hat
        W3 -= LEARNING_RATE / (np.sqrt(vW3_hat) + EPSILON) * mW3_hat
        b3 -= LEARNING_RATE / (np.sqrt(vb3_hat) + EPSILON) * mb3_hat
        W4 -= LEARNING_RATE / (np.sqrt(vW4_hat) + EPSILON) * mW4_hat
        b4 -= LEARNING_RATE / (np.sqrt(vb4_hat) + EPSILON) * mb4_hat
    
    avg_loss = total_loss / (len(indices) // BATCH_SIZE)
    print(f"End of Epoch {epoch}, Loss: {avg_loss:.4f}")

Epoch 0, Batch 0: Loss: 0.6940
Epoch 0, Batch 100: Loss: 0.6912
Epoch 0, Batch 200: Loss: 0.6929
Epoch 0, Batch 300: Loss: 0.6924
Epoch 0, Batch 400: Loss: 0.6899
Epoch 0, Batch 500: Loss: 0.6900
Epoch 0, Batch 600: Loss: 0.6912
Epoch 0, Batch 700: Loss: 0.6900
Epoch 0, Batch 800: Loss: 0.6893
Epoch 0, Batch 900: Loss: 0.6904
Epoch 0, Batch 1000: Loss: 0.6947
Epoch 0, Batch 1100: Loss: 0.6884
Epoch 0, Batch 1200: Loss: 0.6862
Epoch 0, Batch 1300: Loss: 0.6897
Epoch 0, Batch 1400: Loss: 0.6870
Epoch 0, Batch 1500: Loss: 0.6946
Epoch 0, Batch 1600: Loss: 0.6850
Epoch 0, Batch 1700: Loss: 0.6914
Epoch 0, Batch 1800: Loss: 0.6908
Epoch 0, Batch 1900: Loss: 0.6904
Epoch 0, Batch 2000: Loss: 0.6915
Epoch 0, Batch 2100: Loss: 0.6887
Epoch 0, Batch 2200: Loss: 0.6895
Epoch 0, Batch 2300: Loss: 0.6865
Epoch 0, Batch 2400: Loss: 0.6828
Epoch 0, Batch 2500: Loss: 0.6881
Epoch 0, Batch 2600: Loss: 0.6897
Epoch 0, Batch 2700: Loss: 0.6894
Epoch 0, Batch 2800: Loss: 0.6862
Epoch 0, Batch 2900: Loss:

KeyboardInterrupt: 